In [ ]:
# Cell 1: Install & Download Data from Kaggle
# ══════════════════════════════════════════
!pip install -q kaggle

# Upload kaggle.json (API key)
# Lấy từ: https://www.kaggle.com/settings → Create New Token → download kaggle.json
from google.colab import files
print("Upload file kaggle.json:")
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d minhnh2107/casiafasd -p /content/
!unzip -q /content/casiafasd.zip -d /content/Casia-fasd/

# Check structure
import os
DATA_ROOT = "/content/Casia-fasd"
for root, dirs, fls in os.walk(DATA_ROOT):
    level = root.replace(DATA_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    if level < 3:
        for d in dirs:
            print(f"{indent}  {d}/")
    if level >= 2:
        print(f"{indent}  ({len(fls)} files)")
        break

In [ ]:
# Cell 2: Config
# ══════════════════════════════════════════
import torch

# Auto-detect train/test paths
DATA_ROOT = "/content/Casia-fasd"

# Tìm đúng path (handle cấu trúc zip khác nhau)
for candidate in [
    (f"{DATA_ROOT}/train_img", f"{DATA_ROOT}/test_img"),
    (f"{DATA_ROOT}/Casia-fasd/train_img", f"{DATA_ROOT}/Casia-fasd/test_img"),
    (f"{DATA_ROOT}/casiafasd/train_img", f"{DATA_ROOT}/casiafasd/test_img"),
]:
    if os.path.exists(candidate[0]):
        TRAIN_ROOT, TEST_ROOT = candidate
        break
else:
    # Fallback: tìm tự động
    for root, dirs, _ in os.walk(DATA_ROOT):
        if "train_img" in dirs:
            TRAIN_ROOT = os.path.join(root, "train_img")
            TEST_ROOT = os.path.join(root, "test_img")
            break

print(f"Train: {TRAIN_ROOT}")
print(f"Test:  {TEST_ROOT}")

CONF = {
    "lr": 0.01,
    "milestones": [10, 15, 22],
    "gamma": 0.1,
    "epochs": 25,
    "momentum": 0.9,
    "weight_decay": 5e-4,
    "batch_size": 128,

    "num_classes": 2,
    "img_channel": 3,
    "embedding_size": 128,
    "img_size": (80, 80),
    "ft_size": (10, 10),
    "use_ft": True,

    "cls_weight": 1.0,
    "ft_weight": 0.5,

    "train_root": TRAIN_ROOT,
    "test_root": TEST_ROOT,
    "num_workers": 2,

    "save_dir": "/content/saved_models",
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
}

os.makedirs(CONF["save_dir"], exist_ok=True)
device = torch.device(CONF["device"])
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: Dataset & DataLoader
# ══════════════════════════════════════════
import numpy as np
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms


def get_label_from_filename(filename):
    """'10_4.avi_25_fake.jpg' → 0, '10_2.avi_100_real.jpg' → 1"""
    tag = Path(filename).stem.rsplit("_", 1)[-1].lower()
    if tag == "real":
        return 1
    elif tag == "fake":
        return 0
    return None


def find_color_dir(base_dir):
    base = Path(base_dir)
    if (base / "color").is_dir():
        return base / "color"
    for sub in base.iterdir():
        if sub.is_dir() and (sub / "color").is_dir():
            return sub / "color"
    return base  # fallback


class FASDataset(Dataset):
    def __init__(self, root_dir, img_size=(80, 80), ft_size=None,
                 transform=None, is_train=True):
        self.img_size = img_size
        self.ft_size = ft_size

        if transform is not None:
            self.transform = transform
        elif is_train:
            self.transform = transforms.Compose([
                transforms.Resize(img_size),
                transforms.ColorJitter(brightness=0.4, contrast=0.4,
                                       saturation=0.4, hue=0.1),
                transforms.RandomRotation(10),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize([0.5]*3, [0.5]*3),
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize(img_size),
                transforms.ToTensor(),
                transforms.Normalize([0.5]*3, [0.5]*3),
            ])

        self.raw_transform = transforms.Compose([
            transforms.Resize(img_size), transforms.ToTensor(),
        ])

        self.samples = []
        exts = {'.png', '.jpg', '.jpeg', '.bmp'}
        color_dir = find_color_dir(Path(root_dir))

        skipped = 0
        for f in sorted(color_dir.iterdir()):
            if not f.is_file() or f.suffix.lower() not in exts:
                continue
            label = get_label_from_filename(f.name)
            if label is None:
                skipped += 1
                continue
            self.samples.append((str(f), label))

        n_real = sum(1 for _, l in self.samples if l == 1)
        n_fake = sum(1 for _, l in self.samples if l == 0)
        print(f"  [{Path(root_dir).name}] {len(self.samples)} samples "
              f"(real={n_real}, fake={n_fake}, skip={skipped})")

    def __len__(self):
        return len(self.samples)

    def _compute_ft_map(self, img_tensor):
        gray = 0.299*img_tensor[0] + 0.587*img_tensor[1] + 0.114*img_tensor[2]
        f_shift = np.fft.fftshift(np.fft.fft2(gray.numpy()))
        mag = np.log1p(np.abs(f_shift))
        if mag.max() > mag.min():
            mag = (mag - mag.min()) / (mag.max() - mag.min())
        mag_pil = Image.fromarray((mag * 255).astype(np.uint8))
        mag_pil = mag_pil.resize((self.ft_size[1], self.ft_size[0]), Image.BILINEAR)
        return torch.tensor(np.array(mag_pil, dtype=np.float32) / 255.0).unsqueeze(0)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        img_t = self.transform(img)
        if self.ft_size is not None:
            ft = self._compute_ft_map(self.raw_transform(img))
            return img_t, ft, label
        return img_t, label


ft_size = tuple(CONF["ft_size"]) if CONF["use_ft"] else None

print("Loading data...")
train_loader = DataLoader(
    FASDataset(CONF["train_root"], tuple(CONF["img_size"]), ft_size, is_train=True),
    batch_size=CONF["batch_size"], shuffle=True, num_workers=CONF["num_workers"],
    pin_memory=True, drop_last=True)
test_loader = DataLoader(
    FASDataset(CONF["test_root"], tuple(CONF["img_size"]), ft_size=None, is_train=False),
    batch_size=CONF["batch_size"], shuffle=False, num_workers=CONF["num_workers"],
    pin_memory=True)

batch = next(iter(train_loader))
print(f"✓ Batch: img={batch[0].shape}, ft={batch[1].shape}, labels={batch[2][:8]}"
      if CONF["use_ft"] else f"✓ Batch: img={batch[0].shape}, labels={batch[1][:8]}")

In [ ]:
# Cell 4: MobileNeXt Backbone + MultiFTNet
# ══════════════════════════════════════════
import torch.nn as nn
import torch.nn.functional as F
import math


def _make_divisible(v, divisor, min_value=None):
    """
    This function is taken from the original tf repo.
    It ensures that all layers have a channel number that is divisible by 8
    It can be seen here:
    https://github.com/tensorflow/models/blob/master/research/slim/nets/mobilenet/mobilenet.py
    :param v:
    :param divisor:
    :param min_value:
    :return:
    """
    if min_value is None:
        min_value = divisor
    new_v = max(min_value, int(v + divisor / 2) // divisor * divisor)
    # Make sure that round down does not go down by more than 10%.
    if new_v < 0.9 * v:
        new_v += divisor
    return new_v


def conv_3x3_bn(inp, oup, stride):
    return nn.Sequential(
        nn.Conv2d(inp, oup, 3, stride, 1, bias=False),
        nn.BatchNorm2d(oup),
        nn.ReLU6(inplace=True)
    )


def conv_1x1_bn(inp, oup):
    return nn.Sequential(
        nn.Conv2d(inp, oup, 1, 1, 0, bias=False),
        nn.BatchNorm2d(oup),
        nn.ReLU6(inplace=True)
    )

def group_conv_1x1_bn(inp, oup, expand_ratio):
    hidden_dim = oup // expand_ratio
    return nn.Sequential(
        nn.Conv2d(inp, hidden_dim, 1, 1, 0, groups=hidden_dim, bias=False),
        nn.BatchNorm2d(hidden_dim),
        nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
        nn.BatchNorm2d(oup),
        nn.ReLU6(inplace=True)
    )

class SGBlock(nn.Module):
    def __init__(self, inp, oup, stride, expand_ratio, keep_3x3=False):
        super(SGBlock, self).__init__()
        assert stride in [1, 2]

        hidden_dim = inp // expand_ratio
        if hidden_dim < oup / 6.:
            hidden_dim = math.ceil(oup / 6.)
            hidden_dim = _make_divisible(hidden_dim, 16)# + 16

        #self.relu = nn.ReLU6(inplace=True)
        self.identity = False
        self.identity_div = 1
        self.expand_ratio = expand_ratio
        if expand_ratio == 2:
            self.conv = nn.Sequential(
                # dw
                nn.Conv2d(inp, inp, 3, 1, 1, groups=inp, bias=False),
                nn.BatchNorm2d(inp),
                nn.ReLU6(inplace=True),
                # pw-linear
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                # pw-linear
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
                nn.ReLU6(inplace=True),
                # dw
                nn.Conv2d(oup, oup, 3, stride, 1, groups=oup, bias=False),
                nn.BatchNorm2d(oup),
            )
        elif inp != oup and stride == 1 and keep_3x3 == False:
            self.conv = nn.Sequential(
                # pw-linear
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                # pw-linear
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
                nn.ReLU6(inplace=True),
            )
        elif inp != oup and stride == 2 and keep_3x3==False:
            self.conv = nn.Sequential(
                # pw-linear
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                # pw-linear
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
                nn.ReLU6(inplace=True),
                # dw
                nn.Conv2d(oup, oup, 3, stride, 1, groups=oup, bias=False),
                nn.BatchNorm2d(oup),
            )
        else:
            if keep_3x3 == False:
                self.identity = True
            self.conv = nn.Sequential(
                # dw
                nn.Conv2d(inp, inp, 3, 1, 1, groups=inp, bias=False),
                nn.BatchNorm2d(inp),
                nn.ReLU6(inplace=True),
                # pw
                nn.Conv2d(inp, hidden_dim, 1, 1, 0, bias=False),
                nn.BatchNorm2d(hidden_dim),
                #nn.ReLU6(inplace=True),
                # pw
                nn.Conv2d(hidden_dim, oup, 1, 1, 0, bias=False),
                nn.BatchNorm2d(oup),
                nn.ReLU6(inplace=True),
                # dw
                nn.Conv2d(oup, oup, 3, 1, 1, groups=oup, bias=False),
                nn.BatchNorm2d(oup),
            )

    def forward(self, x):
        out = self.conv(x)

        if self.identity:
            shape = x.shape
            id_tensor = x[:,:shape[1]//self.identity_div,:,:]
            # id_tensor = torch.cat([x[:,:shape[1]//self.identity_div,:,:],torch.zeros(shape)[:,shape[1]//self.identity_div:,:,:].cuda()],dim=1)
            # import pdb; pdb.set_trace()
            out[:,:shape[1]//self.identity_div,:,:] = out[:,:shape[1]//self.identity_div,:,:] + id_tensor
            return out #+ x
        else:
            return out

class MXNet(nn.Module):
    def __init__(self, num_classes=1000, width_mult=1., in_channels=3):
        super(MXNet, self).__init__()
        # setting of SGB blocks
        self.cfgs = [
            # t, c, n, s  for ImageNet
            #[2,  96, 1, 2],
            #[6, 144, 1, 1],
            #[6, 192, 3, 2],
            #[6, 288, 3, 2],
            #[6, 384, 4, 1],
            #[6, 576, 4, 2],
            #[6, 960, 3, 1],
            #[6,1280, 1, 1],

            # CIFAR 10 and 100 high parameter version
            #[2,   96, 1, 1],
            #[6,  144, 1, 1],
            #[6,  192, 3, 1],
            #[6,  288, 3, 2],
            #[6,  384, 4, 1],
            #[6,  576, 4, 2],
            #[6,  960, 3, 1],
            #[6, 1280, 1, 1],

            # CIFAR 10 and 100 low parameter version
            [2,   64, 1, 1],  
            [6,   96, 1, 1],  
            [6,  128, 3, 2],  
            [6,  192, 2, 1],  
            [6,  256, 3, 2],  
            [6,  384, 2, 1],  
            [6,  512, 1, 1],
        ]
        #self.cfgs = [
        #    # t, c, n, s
        #    [1,  16, 1, 1],
        #    [4,  24, 2, 2],
        #    [4,  32, 3, 2],
        #    [4,  64, 3, 2],
        #    [4,  96, 4, 1],
        #    [4, 160, 3, 2],
        #    [4, 320, 1, 1],
        #]

        # building first layer
        input_channel = _make_divisible(32 * width_mult, 4 if width_mult == 0.1 else 8)
        layers = [conv_3x3_bn(in_channels, input_channel, 1)] # stride 2->1 for CIFAR
        # building inverted residual blocks
        block = SGBlock
        for t, c, n, s in self.cfgs:
            output_channel = _make_divisible(c * width_mult, 4 if width_mult == 0.1 else 8)
            if c == 1280 and width_mult < 1:
                output_channel = 1280
            layers.append(block(input_channel, output_channel, s, t, n==1 and s==1))
            input_channel = output_channel
            for i in range(n-1):
                layers.append(block(input_channel, output_channel, 1, t))
                input_channel = output_channel
        self.features = nn.Sequential(*layers)
        # building last several layers
        input_channel = output_channel
        output_channel = _make_divisible(input_channel, 4) # if width_mult == 0.1 else 8) if width_mult > 1.0 else input_channel
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
                nn.Dropout(0.2),
                nn.Linear(output_channel, num_classes)
                )

        self._initialize_weights()

    def forward(self, x):
        x = self.features(x)
        #x = self.conv(x)
        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                n = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, math.sqrt(2. / n))
                if m.bias is not None:
                    m.bias.data.zero_()
            elif isinstance(m, nn.BatchNorm2d):
                m.weight.data.fill_(1)
                m.bias.data.zero_()
            elif isinstance(m, nn.Linear):
                m.weight.data.normal_(0, 0.01)
                m.bias.data.zero_()


class FTGenerator(nn.Module):
    """Generates Fourier feature map from intermediate backbone features."""

    def __init__(self, in_channels=128, out_channels=1):
        super(FTGenerator, self).__init__()
        self.ft = nn.Sequential(
            nn.Conv2d(in_channels, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.ft(x)


class MultiFTNet(nn.Module):
    """Multi-task FAS model.

    Architecture:
        Input → early_features (128ch) → late_features (512ch) → classification
                        ↓
                   FTGenerator → Fourier feature map

    Args:
        num_classes: number of output classes (2 for CASIA-FASD: real/fake)
        img_channel: input image channels
        embedding_size: FC embedding dimension before classifier
        ft_size: (H, W) of output Fourier feature map. None = no resize.
    """

    def __init__(self, num_classes=2, img_channel=3, embedding_size=128, ft_size=None, **kwargs):
        super(MultiFTNet, self).__init__()
        self.num_classes = num_classes
        self.ft_size = ft_size

        # Build MobileNeXt backbone
        backbone = MXNet(num_classes=num_classes, in_channels=img_channel)

        # Split features at 128-channel boundary
        features = list(backbone.features.children())
        self.early_features = nn.Sequential(*features[:6])   # output: 128 channels
        self.late_features = nn.Sequential(*features[6:])     # output: 512 channels
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # Classification head
        self.linear = nn.Linear(512, embedding_size, bias=False)
        self.bn = nn.BatchNorm1d(embedding_size)
        self.drop = nn.Dropout(p=0.2)
        self.prob = nn.Linear(embedding_size, num_classes, bias=False)

        # Fourier feature map head
        self.FTGenerator = FTGenerator(in_channels=128, out_channels=1)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.001)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Early features → 128 channels
        x_early = self.early_features(x)

        # Late features → classification
        x_late = self.late_features(x_early)
        x_late = self.avgpool(x_late)
        x_late = x_late.view(x_late.size(0), -1)
        x_late = self.linear(x_late)
        x_late = self.bn(x_late)
        x_late = self.drop(x_late)
        cls = self.prob(x_late)

        if self.training:
            ft = self.FTGenerator(x_early)
            # Resize FT map to target size if specified
            if self.ft_size is not None:
                ft = F.interpolate(ft, size=self.ft_size, mode='bilinear', align_corners=False)
            return cls, ft
        else:
            return cls


model = MultiFTNet(
    num_classes=CONF["num_classes"], img_channel=CONF["img_channel"],
    embedding_size=CONF["embedding_size"], ft_size=ft_size,
).to(device)

print(f"✓ MultiFTNet | Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 5: Train
# ══════════════════════════════════════════
import time
from tqdm.notebook import tqdm

def evaluate(model, loader, device):
    model.eval()
    tp = fp = tn = fn = 0
    with torch.no_grad():
        for batch in loader:
            imgs, labels = batch[0].to(device), batch[-1].to(device)
            _, preds = torch.max(model(imgs), 1)
            for p, g in zip(preds, labels):
                if g==1 and p==1: tp+=1
                elif g==0 and p==1: fp+=1
                elif g==0 and p==0: tn+=1
                else: fn+=1
    total = tp+fp+tn+fn
    acc = (tp+tn)/total if total else 0
    apcer = fp/(tn+fp) if (tn+fp) else 0
    bpcer = fn/(fn+tp) if (fn+tp) else 0
    model.train()
    return acc, apcer, bpcer, (apcer+bpcer)/2

cls_crit = nn.CrossEntropyLoss()
ft_crit = nn.MSELoss()
optim_ = torch.optim.SGD(model.parameters(), lr=CONF["lr"],
                          momentum=CONF["momentum"], weight_decay=CONF["weight_decay"])
sched = torch.optim.lr_scheduler.MultiStepLR(optim_, CONF["milestones"], CONF["gamma"])

history = {"train_loss":[], "train_acc":[], "test_acc":[], "apcer":[], "bpcer":[], "acer":[]}
best_acer = 1.0

for epoch in range(CONF["epochs"]):
    model.train()
    rloss = rcorr = rtotal = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONF['epochs']}")

    for batch_data in pbar:
        if CONF["use_ft"]:
            imgs, fts, labels = batch_data
            fts = fts.to(device)
        else:
            imgs, labels = batch_data
        imgs, labels = imgs.to(device), labels.to(device)

        optim_.zero_grad()
        if CONF["use_ft"]:
            cls_out, ft_out = model(imgs)
            loss = CONF["cls_weight"]*cls_crit(cls_out, labels) + CONF["ft_weight"]*ft_crit(ft_out, fts)
        else:
            cls_out = model(imgs)
            loss = cls_crit(cls_out, labels)

        loss.backward()
        optim_.step()

        _, preds = torch.max(cls_out, 1)
        rtotal += labels.size(0)
        rcorr += (preds==labels).sum().item()
        rloss += loss.item()
        pbar.set_postfix(loss=f"{rloss/(pbar.n+1):.4f}", acc=f"{rcorr/rtotal:.4f}")

    sched.step()
    eloss = rloss/len(train_loader)
    eacc = rcorr/rtotal
    acc, apcer, bpcer, acer = evaluate(model, test_loader, device)

    history["train_loss"].append(eloss); history["train_acc"].append(eacc)
    history["test_acc"].append(acc); history["apcer"].append(apcer)
    history["bpcer"].append(bpcer); history["acer"].append(acer)

    print(f"  Loss:{eloss:.4f} TrainAcc:{eacc:.4f} TestAcc:{acc:.4f} "
          f"APCER:{apcer:.4f} BPCER:{bpcer:.4f} ACER:{acer:.4f}")

    if acer < best_acer:
        best_acer = acer
        torch.save({"epoch":epoch, "model_state_dict":model.state_dict(),
                     "acer":acer, "acc":acc},
                    f"{CONF['save_dir']}/best_model.pth")
        print(f"  ★ Best saved (ACER: {acer:.4f})")

print(f"\n✅ Done! Best ACER: {best_acer:.4f}")